<a href="https://colab.research.google.com/github/MiguelDantasdeOliveira/Trabalho-POO/blob/master/Trabalhofinal1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [41]:
!pip install -q streamlit faiss-cpu sentence-transformers transformers accelerate
!wget -q -nc https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar -xvzf ngrok-v3-stable-linux-amd64.tgz
!mv ngrok /usr/local/bin
#Instala dependências para NLP, RAG e interface.
#Baixa e instala o ngrok para expor a interface Streamlit na web.

ngrok


In [42]:
!pip install openai==0.28

In [43]:
!pip install numpy

In [44]:
!pip install pandas openpyxl

In [45]:
import pandas as pd
df = pd.read_excel("Perguntas_IA_Sistema_RAG.xlsx")
documentos = df["Pergunta"].dropna().tolist()
print(f"Carregados {len(documentos)} documentos para indexar.")
#Carrega o conteúdo da planilha como base documental.

Carregados 10 documentos para indexar.


In [46]:
from transformers import pipeline
gerador = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto", max_new_tokens=300)

def gerar_resposta_rag(pergunta, k=3):
    trechos = buscar_similares(pergunta, k)
    contexto = "\n".join(trechos)
    prompt = f"""### Instrução:
Com base nos textos abaixo, responda a pergunta do usuário.

### Contexto:
{contexto}

### Pergunta:
{pergunta}

### Resposta:"""

    resposta = gerador(prompt)[0]["generated_text"].split("### Resposta:")[-1].strip()
    return resposta
#Pipeline de geração usando o modelo TinyLlama local.

Device set to use cpu


In [47]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')  # leve e eficiente
embeddings = model.encode(documentos, convert_to_numpy=True)
print(f"Embeddings gerados: shape {embeddings.shape}")
#Gera embeddings vetoriais dos documentos.
#all-MiniLM-L6-v2 é leve e eficiente.

Embeddings gerados: shape (10, 384)


In [48]:
import faiss

dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)
print(f"Índice FAISS criado com {index.ntotal} vetores.")
#Cria um índice vetorial simples (sem compressão, flat).
#dim é a dimensão dos embeddings (normalmente 384 para MiniLM).

Índice FAISS criado com 10 vetores.


In [49]:
def buscar_similares(query, k=3):
    emb_query = model.encode([query], convert_to_numpy=True)
    D, I = index.search(emb_query, k)
    return [(documentos[i], D[0][idx]) for idx, i in enumerate(I[0])]
#Retorna os k documentos mais similares a uma pergunta.

In [57]:
pergunta = "O que é Inteligência Artificial e como ela funciona?"
resposta = gerar_resposta_rag(pergunta)
print("Pergunta:", pergunta)
print("Resposta:", resposta)

Pergunta: O que é Inteligência Artificial e como ela funciona?
Resposta: ('Inteligência Artificial é a capacidade de máquinas realizar tarefas humanas. A IA funciona por meio de algoritmos que aprendem com dados. Redes neurais artificiais são inspiradas no cérebro humano.\n\n### Exemplo:\nConsidere um caso em que uma empresa precisa identificar os clientes que possuem uma necessidade de produtos específicos. A IA pode ser utilizada para identificar esses clientes, utilizando dados de marketing e de compras. A IA aprende com dados e, com o tempo, pode identificar novas necessidades e, portanto, identificar novos clientes.\n\n### Conclusão:\nA IA é uma das principais tecnologias de nuvem e é utilizada em vários setores, como a saúde, a economia e a segurança. A IA é uma tecnologia que tem grande potencial para transformar a vida de pessoas e empresas.', ['Inteligência Artificial é a capacidade de máquinas realizarem tarefas humanas.', 'A IA funciona por meio de algoritmos que aprendem co

In [53]:
import streamlit as st

st.title("Sistema RAG - Perguntas e Respostas")

pergunta = st.text_input("Digite sua pergunta:")

if pergunta:
    resposta = gerar_resposta_rag(pergunta)
    st.markdown("### Resposta:")
    st.write(resposta)

2025-05-23 23:47:46.114 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.118 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.120 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.121 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.122 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.123 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.125 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:47:46.126 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

In [54]:
import streamlit as st
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

# ---------------------------
# Carregar modelo de embeddings e LLM
# ---------------------------
modelo_embedding = SentenceTransformer("all-MiniLM-L6-v2")  # Leve, ideal para Colab
gerador = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto", max_new_tokens=300)

# ---------------------------
# Simular base vetorial FAISS
# ---------------------------
# Suponha que esses são os textos indexados do seu .xlsx
textos = [
    "Inteligência Artificial é a capacidade de máquinas realizarem tarefas humanas.",
    "A IA funciona por meio de algoritmos que aprendem com dados.",
    "Redes neurais artificiais são inspiradas no cérebro humano."
]

# Gerar embeddings
embeddings = modelo_embedding.encode(textos, convert_to_numpy=True).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

# ---------------------------
# Função de busca no índice + geração de resposta
# ---------------------------
def buscar_similares(pergunta, k=3):
    embed = modelo_embedding.encode([pergunta], convert_to_numpy=True).astype("float32")
    dists, indices = index.search(embed, k)
    return [textos[i] for i in indices[0]]
#Interface amigável onde o usuário insere perguntas.
#Usa st.write() para exibir a resposta e os trechos base.

def gerar_resposta_rag(pergunta, k=3):
    trechos = buscar_similares(pergunta, k)
    contexto = "\n".join(trechos)
    prompt = f"""### Instrução:
Com base nos textos abaixo, responda à pergunta do usuário.

### Contexto:
{contexto}

### Pergunta:
{pergunta}

### Resposta:"""

    resposta = gerador(prompt)[0]["generated_text"].split("### Resposta:")[-1].strip()
    return resposta, trechos

# ---------------------------
# Interface Streamlit
# ---------------------------
st.set_page_config(page_title="Sistema RAG", layout="centered")
st.title("🤖 Sistema RAG com TinyLlama + FAISS")

pergunta = st.text_input("Digite sua pergunta:")
if st.button("Responder") and pergunta:
    with st.spinner("Gerando resposta..."):
        resposta, trechos = gerar_resposta_rag(pergunta)
    st.subheader("📌 Resposta:")
    st.write(resposta)
    st.subheader("📚 Trechos usados:")
    for t in trechos:
        st.markdown(f"- {t}")

Device set to use cpu
2025-05-23 23:48:02.205 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.210 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.214 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.219 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.222 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.226 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.229 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-23 23:48:02.232 Thread 'MainThread': missing ScriptRunContext! This warning can be ignor

In [55]:
!ngrok config add-authtoken SEU_TOKEN_NGROK
#Permite expor o app Streamlit fora do Colab.

Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [56]:
%%writefile app.py
import streamlit as st
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import pipeline

modelo_embedding = SentenceTransformer("all-MiniLM-L6-v2")
gerador = pipeline("text-generation", model="TinyLlama/TinyLlama-1.1B-Chat-v1.0", device_map="auto", max_new_tokens=300)

textos = [
    "Inteligência Artificial é a capacidade de máquinas realizarem tarefas humanas.",
    "A IA funciona por meio de algoritmos que aprendem com dados.",
    "Redes neurais artificiais são inspiradas no cérebro humano."
]

embeddings = modelo_embedding.encode(textos, convert_to_numpy=True).astype("float32")
dim = embeddings.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(embeddings)

def buscar_similares(pergunta, k=3):
    embed = modelo_embedding.encode([pergunta], convert_to_numpy=True).astype("float32")
    dists, indices = index.search(embed, k)
    return [textos[i] for i in indices[0]]

def gerar_resposta_rag(pergunta, k=3):
    trechos = buscar_similares(pergunta, k)
    contexto = "\n".join(trechos)
    prompt = f"""### Instrução:
Com base nos textos abaixo, responda à pergunta do usuário.

### Contexto:
{contexto}

### Pergunta:
{pergunta}

### Resposta:"""

    resposta = gerador(prompt)[0]["generated_text"].split("### Resposta:")[-1].strip()
    return resposta, trechos

st.set_page_config(page_title="Sistema RAG", layout="centered")
st.title("🤖 Sistema RAG com TinyLlama + FAISS")

pergunta = st.text_input("Digite sua pergunta:")
if st.button("Responder") and pergunta:
    with st.spinner("Gerando resposta..."):
        resposta, trechos = gerar_resposta_rag(pergunta)
    st.subheader("📌 Resposta:")
    st.write(resposta)
    st.subheader("📚 Trechos usados:")
    for t in trechos:
        st.markdown(f"- {t}")
#Junta tudo em um único arquivo executável com streamlit run app.py.
#Útil para organização e reusabilidade.

Overwriting app.py
